In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset

In [2]:
VOCAB_SIZE = 12000
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 6          

BATCH_SIZE = 64
EPOCHS = 5
MAX_LEN = 60

In [3]:
print("Loading dair-ai/emotion...")
dataset = load_dataset("dair-ai/emotion")
print(dataset)

Loading dair-ai/emotion...


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [4]:
X_train_raw = list(dataset["train"]["text"])
y_train_strings = list(dataset["train"]["label"])

In [15]:
label_to_id = {
    "sadness": 0,
    "enjoy": 1,
    "love": 2,
    "anger": 3,
    "fear": 4,
    "surprise": 5
}
id_to_label = {v: k for k, v in label_to_id.items()}

In [16]:
def encode_labels(raw_labels):
    """Handles both already-integer labels and string labels, just in case."""
    encoded = []
    for label in raw_labels:
        if isinstance(label, str):
            encoded.append(label_to_id.get(label.lower().strip(), 0))
        else:
            encoded.append(int(label))
    return np.array(encoded, dtype=np.int32)

y_all_label = encode_labels(y_train_strings)

In [17]:
if "validation" in dataset:
    x_val_raw = list(dataset["validation"]["text"])
    y_val_strings = list(dataset["validation"]["label"])
    y_val = encode_labels(y_val_strings)
    y_train = y_all_label

    print(f"Training samples:{len(X_train_raw)}")
    print(f"Validation samples:{len(x_val_raw)}")
else:
    total_len = len(X_train_raw)
    val_size = int(total_len * 0.1)
    split_idx = total_len - val_size

    x_val_raw = X_train_raw[split_idx:]
    X_train_raw = X_train_raw[:split_idx]

    y_val = y_all_label[split_idx:]
    y_train = y_all_label[:split_idx]

    print(f"Training samples:{len(X_train_raw)}")
    print(f"Validation samples:{len(x_val_raw)}")

if "test" in dataset:
    x_test_raw = list(dataset["test"]["text"])
    y_test_strings = list(dataset["test"]["label"])
    y_test = encode_labels(y_test_strings)
    print(f"Test samples:{len(x_test_raw)}")

Training samples:16000
Validation samples:2000
Test samples:2000


In [18]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_raw)

X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
x_val_seq = tokenizer.texts_to_sequences(x_val_raw)

X_train = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
x_val = pad_sequences(x_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")

if "test" in dataset:
    x_test_seq = tokenizer.texts_to_sequences(x_test_raw)
    x_test = pad_sequences(x_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print("X_train shape:", X_train.shape)
print("x_val shape:", x_val.shape)

X_train shape: (16000, 60)
x_val shape: (2000, 60)


In [19]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    LSTM(HIDDEN_DIM),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(OUTPUT_DIM, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
history = model.fit(
    X_train, y_train,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS
)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 74s 259ms/step - accuracy: 0.3117 - loss: 1.6091 - val_accuracy: 0.3520 - val_loss: 1.5809
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 83s 261ms/step - accuracy: 0.3238 - loss: 1.5873 - val_accuracy: 0.3520 - val_loss: 1.5799
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 66s 263ms/step - accuracy: 0.3266 - loss: 1.5850 - val_accuracy: 0.3520 - val_loss: 1.5809
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 69s 276ms/step - accuracy: 0.3306 - loss: 1.5828 - val_accuracy: 0.3520 - val_loss: 1.5822
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 73s 290ms/step - accuracy: 0.3315 - loss: 1.5801 - val_accuracy: 0.3520 - val_loss: 1.5795


In [21]:
if "test" in dataset:
    test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=BATCH_SIZE)
    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")

32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 111ms/step - accuracy: 0.3475 - loss: 1.5603
Test loss: 1.5603
Test accuracy: 0.3475


In [22]:

emotion_to_sentiment = {
    "joy": "Positive",
    "love": "Positive",
    "surprise": "Positive",
    "sadness": "Negative",
    "anger": "Negative",
    "fear": "Negative"
}

def predict_emotion(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    probs = model.predict(padded, verbose=0)[0]
    pred_id = int(np.argmax(probs))
    label = id_to_label[pred_id]
    confidence = float(probs[pred_id]) * 100
    sentiment = emotion_to_sentiment.get(label, "Neutral")
    return label, sentiment, confidence

def show_prediction(text):
    label, sentiment, confidence = predict_emotion(text)
    print("=" * 32)
    print("      MODEL PREDICTION")
    print("=" * 32)
    print(f"Input Text: {text}")
    print(f"Predicted Emotion: {label}")
    print(f"Sentiment: {sentiment}")
    print(f"Confidence: {confidence:.2f} %")

In [24]:
show_prediction("I am excited about my new achievement.")

      MODEL PREDICTION
Input Text: I am excited about my new achievement.
Predicted Emotion: enjoy
Sentiment: Neutral
Confidence: 33.39 %
